# Pandas Essentials for Data Analysts

This notebook covers the main pandas operations every data analyst should know. It includes data loading, inspection, selection, transformation, grouping, missing value handling, date processing, string handling, merging, and export.

## 1. Setup and import

Start by importing pandas and setting a display option for better output readability.

In [1]:
import pandas as pd
from io import StringIO

pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 120)

print('pandas version:', pd.__version__)

pandas version: 3.0.3


## 2. Load data from common sources

Typical data sources are CSV, Excel, and JSON. Use pandas readers to load data into a DataFrame.

In [2]:
csv_data = '''
order_id,customer,product,quantity,price,order_date,status
1001,Alice,Widget,4,20.5,2024-06-01,Delivered
1002,Bob,Gadget,2,15.0,2024-06-03,Returned
1003,Charlie,Widget,1,20.5,2024-06-05,Processing
1004,Dana,Doohickey,3,12.0,2024-06-07,Delivered
1005,Eli,Gadget,5,15.0,2024-06-10,Delivered
'''

df = pd.read_csv(StringIO(csv_data), parse_dates=['order_date'])
df.head()

,order_id,customer,product,quantity,price,order_date,status
0,1001,Alice,Widget,4,20.5,2024-06-01,Delivered
1,1002,Bob,Gadget,2,15.0,2024-06-03,Returned
2,1003,Charlie,Widget,1,20.5,2024-06-05,Processing
3,1004,Dana,Doohickey,3,12.0,2024-06-07,Delivered
4,1005,Eli,Gadget,5,15.0,2024-06-10,Delivered


## 3. Inspect the DataFrame

Use these methods to understand the dataset shape, types, and summary statistics.

In [3]:
print('Shape:', df.shape)
print('Columns:', df.columns.tolist())

df.info()

display(df.describe(include='all'))

Shape: (5, 7)
Columns: ['order_id', 'customer', 'product', 'quantity', 'price', 'order_date', 'status']
<class 'pandas.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 7 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   order_id    5 non-null      int64         
 1   customer    5 non-null      str           
 2   product     5 non-null      str           
 3   quantity    5 non-null      int64         
 4   price       5 non-null      float64       
 5   order_date  5 non-null      datetime64[us]
 6   status      5 non-null      str           
dtypes: datetime64[us](1), float64(1), int64(2), str(3)
memory usage: 412.0 bytes


,order_id,customer,product,quantity,price,order_date,status
count,5.000000,5,5,5.000000,5.00000,5,5
unique,NaN,5,3,NaN,NaN,NaN,3
top,NaN,Alice,Widget,NaN,NaN,NaN,Delivered
freq,NaN,1,2,NaN,NaN,NaN,3
mean,1003.000000,NaN,NaN,3.000000,16.60000,2024-06-05 04:48:00,NaN
min,1001.000000,NaN,NaN,1.000000,12.00000,2024-06-01 00:00:00,NaN
25%,1002.000000,NaN,NaN,2.000000,15.00000,2024-06-03 00:00:00,NaN
50%,1003.000000,NaN,NaN,3.000000,15.00000,2024-06-05 00:00:00,NaN
75%,1004.000000,NaN,NaN,4.000000,20.50000,2024-06-07 00:00:00,NaN
max,1005.000000,NaN,NaN,5.000000,20.50000,2024-06-10 00:00:00,NaN


## 4. Select columns and rows

Use `[]`, `.loc`, and `.iloc` to slice data. Boolean indexing is the primary way to filter rows.

In [4]:
# Select a single column
product_series = df['product']

print(product_series.head(), "\n")

# Select multiple columns
df[['customer', 'order_date', 'status']].head()

# Select rows by position
df.iloc[1:4]  # second through fourth rows

# Select rows by label and column names
df.loc[df['status'] == 'Delivered', ['order_id', 'customer', 'status']]

# Select rows by position
df.iloc[1:4]  # second through fourth rows

# Select rows by label and column names
df.loc[df['status'] == 'Delivered', ['order_id', 'customer', 'status']]

0       Widget
1       Gadget
2       Widget
3    Doohickey
4       Gadget
Name: product, dtype: str 



,order_id,customer,status
0,1001,Alice,Delivered
3,1004,Dana,Delivered
4,1005,Eli,Delivered


## 5. Add and transform columns

Create derived columns with vectorized arithmetic and mapping operations.

In [5]:
df['total'] = df['quantity'] * df['price']
df['order_month'] = df['order_date'].dt.to_period('M')

status_map = {'Delivered': 'Complete', 'Returned': 'Failed', 'Processing': 'In Progress'}
df['status_label'] = df['status'].map(status_map)

df.head()

,order_id,customer,product,quantity,price,order_date,status,total,order_month,status_label
0,1001,Alice,Widget,4,20.5,2024-06-01,Delivered,82.0,2024-06,Complete
1,1002,Bob,Gadget,2,15.0,2024-06-03,Returned,30.0,2024-06,Failed
2,1003,Charlie,Widget,1,20.5,2024-06-05,Processing,20.5,2024-06,In Progress
3,1004,Dana,Doohickey,3,12.0,2024-06-07,Delivered,36.0,2024-06,Complete
4,1005,Eli,Gadget,5,15.0,2024-06-10,Delivered,75.0,2024-06,Complete


## 6. Grouping and aggregation

Group by one or more keys, then aggregate numeric values and counts.

In [6]:
# Total revenue by product
revenue_by_product = df.groupby('product', as_index=False).agg(
    total_revenue=('total', 'sum'),
    average_quantity=('quantity', 'mean'),
    orders=('order_id', 'count')
)
revenue_by_product

# Revenue by customer and order month
monthly_customer = df.groupby(['customer', 'order_month'], as_index=False)['total'].sum()
monthly_customer

,customer,order_month,total
0,Alice,2024-06,82.0
1,Bob,2024-06,30.0
2,Charlie,2024-06,20.5
3,Dana,2024-06,36.0
4,Eli,2024-06,75.0


## 7. Sort and rank data

Sort rows to identify top values and use rank when needed.

In [7]:
# Sort by total revenue descending
df.sort_values(by='total', ascending=False).head()

# Rank orders by total in the original order
df['total_rank'] = df['total'].rank(method='dense', ascending=False).astype(int)
df[['order_id', 'customer', 'total', 'total_rank']]

,order_id,customer,total,total_rank
0,1001,Alice,82.0,1
1,1002,Bob,30.0,4
2,1003,Charlie,20.5,5
3,1004,Dana,36.0,3
4,1005,Eli,75.0,2


## 8. Handle missing values

Missing data is common. Inspect and fill or drop values depending on the use case.

In [9]:
df_missing = df.copy()
df_missing.loc[2, 'price'] = None
df_missing.loc[4, 'customer'] = None

print('Missing counts:')
print(df_missing.isna().sum())

# Fill numeric missing values with mean and forward-fill strings
df_missing['price'] = df_missing['price'].fillna(df_missing['price'].mean())
df_missing['customer'] = df_missing['customer'].ffill()

df_missing

Missing counts:
order_id        0
customer        1
product         0
quantity        0
price           1
order_date      0
status          0
total           0
order_month     0
status_label    0
total_rank      0
dtype: int64


,order_id,customer,product,quantity,price,order_date,status,total,order_month,status_label,total_rank
0,1001,Alice,Widget,4,20.500,2024-06-01,Delivered,82.0,2024-06,Complete,1
1,1002,Bob,Gadget,2,15.000,2024-06-03,Returned,30.0,2024-06,Failed,4
2,1003,Charlie,Widget,1,15.625,2024-06-05,Processing,20.5,2024-06,In Progress,5
3,1004,Dana,Doohickey,3,12.000,2024-06-07,Delivered,36.0,2024-06,Complete,3
4,1005,Dana,Gadget,5,15.000,2024-06-10,Delivered,75.0,2024-06,Complete,2


## 9. Work with dates

Date columns can be parsed and used to create features such as day of week, month, and business logic filters.

In [10]:
df['order_day'] = df['order_date'].dt.day_name()
df['order_week'] = df['order_date'].dt.isocalendar().week

df[['order_id', 'order_date', 'order_day', 'order_week']].head()

# Filter orders placed in June 2024
june_orders = df[df['order_date'].dt.month == 6]
june_orders

,order_id,customer,product,quantity,price,order_date,status,total,order_month,status_label,total_rank,order_day,order_week
0,1001,Alice,Widget,4,20.5,2024-06-01,Delivered,82.0,2024-06,Complete,1,Saturday,22
1,1002,Bob,Gadget,2,15.0,2024-06-03,Returned,30.0,2024-06,Failed,4,Monday,23
2,1003,Charlie,Widget,1,20.5,2024-06-05,Processing,20.5,2024-06,In Progress,5,Wednesday,23
3,1004,Dana,Doohickey,3,12.0,2024-06-07,Delivered,36.0,2024-06,Complete,3,Friday,23
4,1005,Eli,Gadget,5,15.0,2024-06-10,Delivered,75.0,2024-06,Complete,2,Monday,24


## 10. String operations

Use vectorized string methods for cleaning and extracting text data.

In [11]:
df['product_clean'] = df['product'].str.lower()
df['customer_initial'] = df['customer'].str[0]

df[['product', 'product_clean', 'customer', 'customer_initial']]

,product,product_clean,customer,customer_initial
0,Widget,widget,Alice,A
1,Gadget,gadget,Bob,B
2,Widget,widget,Charlie,C
3,Doohickey,doohickey,Dana,D
4,Gadget,gadget,Eli,E


## 11. Combine and join data

Merging and concatenating are essential when bringing multiple tables together.

In [12]:
customer_data = pd.DataFrame({
    'customer': ['Alice', 'Bob', 'Charlie', 'Dana', 'Eli'],
    'region': ['North', 'South', 'East', 'West', 'South']
})

merged = df.merge(customer_data, on='customer', how='left')
merged[['order_id', 'customer', 'region', 'total']].head()

# Concatenate new rows to the original DataFrame
new_orders = pd.DataFrame([
    {'order_id': 1006, 'customer': 'Fiona', 'product': 'Widget', 'quantity': 2, 'price': 20.5, 'order_date': pd.Timestamp('2024-06-12'), 'status': 'Delivered', 'total': 41.0, 'order_month': pd.Period('2024-06'), 'status_label': 'Complete'}
])
all_orders = pd.concat([df, new_orders], ignore_index=True)
all_orders.tail()

,order_id,customer,product,quantity,price,order_date,status,total,order_month,status_label,total_rank,order_day,order_week,product_clean,customer_initial
1,1002,Bob,Gadget,2,15.0,2024-06-03,Returned,30.0,2024-06,Failed,4.0,Monday,23,gadget,B
2,1003,Charlie,Widget,1,20.5,2024-06-05,Processing,20.5,2024-06,In Progress,5.0,Wednesday,23,widget,C
3,1004,Dana,Doohickey,3,12.0,2024-06-07,Delivered,36.0,2024-06,Complete,3.0,Friday,23,doohickey,D
4,1005,Eli,Gadget,5,15.0,2024-06-10,Delivered,75.0,2024-06,Complete,2.0,Monday,24,gadget,E
5,1006,Fiona,Widget,2,20.5,2024-06-12,Delivered,41.0,2024-06,Complete,NaN,NaN,<NA>,NaN,NaN


## 12. Export results

Save a cleaned or aggregated DataFrame to CSV or Excel for reporting and sharing.

In [13]:
output_csv = 'pandas_essentials_output.csv'
aggregated = df.groupby('status_label', as_index=False)['total'].sum()
aggregated.to_csv(output_csv, index=False)
print('Saved:', output_csv)
aggregated

Saved: pandas_essentials_output.csv


,status_label,total
0,Complete,193.0
1,Failed,30.0
2,In Progress,20.5
